# Numerical Linear Algebra

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 06.03 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/04_numerical_linear_algebra.ipynb)

---

## 🎯 Learning Objective

Learn iterative solvers like conjugate gradient, and understand how condition number and preconditioning affect their convergence.

---

## 📐 Theory

`01.04` solved linear systems $A\mathbf{x}=\mathbf{b}$ with Gaussian elimination — an exact,
**direct** method costing $O(n^3)$ time and needing to store all of $A$. For the huge, often
sparse systems that show up in large-scale ML, that cost is frequently prohibitive. This
notebook covers **iterative** solvers instead: methods that produce increasingly accurate
approximate solutions, each iteration touching $A$ only through matrix-vector products.

### Conjugate Gradient (CG)

For a **symmetric positive-definite** (SPD) matrix $A$ (recall the positive-definiteness
condition from `01.06`/`04.03`), solving $A\mathbf{x}=\mathbf{b}$ is equivalent to minimizing
the quadratic $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top A\mathbf{x} - \mathbf{b}^\top\mathbf{x}$
(its gradient is $A\mathbf{x}-\mathbf{b}$, zero exactly at the solution). Plain gradient descent
(`04.02`) on this quadratic can zig-zag badly when $A$ is ill-conditioned. **Conjugate
Gradient** fixes this by choosing each search direction to be **A-conjugate** to every previous
direction ($\mathbf{p}_i^\top A \mathbf{p}_j = 0$ for $i\ne j$) rather than just locally
steepest — informally, each new step doesn't undo the progress made by earlier steps, the way
gradient descent's oscillating zig-zags can.

The remarkable guarantee: for an $n\times n$ SPD system, **CG converges to the exact solution
in at most $n$ iterations** (in exact arithmetic), because there are only $n$ mutually
A-conjugate directions available in $\mathbb{R}^n$ — once you've stepped along all of them,
you've exactly cancelled out every component of the initial error. In practice, with rounding
error and for large $n$, CG is used as an *iterative* method run for far fewer than $n$ steps,
stopping once the residual $\|A\mathbf{x}_k - \mathbf{b}\|$ is small enough.

### Condition number and convergence speed

CG's convergence rate (before hitting the hard $n$-step guarantee) depends on $A$'s condition
number $\kappa = \lambda_{max}/\lambda_{min}$ (recall `01.06`, `04.03`): the number of
iterations needed for a given accuracy grows roughly like $\sqrt{\kappa}$. A well-conditioned
system converges in a handful of steps; an ill-conditioned one can need many more, even though
both are still bounded by $n$ overall — this is the same "conditioning determines difficulty"
theme as `04.03`'s convex optimization convergence rate, now applied to solving linear systems
directly rather than minimizing a general convex function.

### Preconditioning

If $A$ itself is ill-conditioned, **preconditioning** solves an *equivalent* system with better
conditioning instead: pick an invertible $M$ that approximates $A$ but is cheap to invert, and
solve $M^{-1}A\mathbf{x} = M^{-1}\mathbf{b}$ (or a symmetric variant) rather than the original
system directly. A good preconditioner $M$ makes $M^{-1}A$ close to the identity (well-conditioned)
without requiring you to actually compute the expensive $A^{-1}$. The tradeoff is real: a better
preconditioner speeds up each CG iteration's convergence but costs more to construct and apply,
so practical preconditioning is itself a balancing act, not a free win.

### Why iterative solvers matter at scale

Direct solvers scale as $O(n^3)$ and need $O(n^2)$ storage; an iterative method like CG needs
only matrix-vector products (often $O(n)$ or $O(n\cdot\text{nnz})$ for a sparse matrix with
nnz nonzero entries) and a handful of $O(n)$ vectors, with no requirement to ever form or
store $A^{-1}$. For the huge, often naturally sparse systems that appear in large-scale
optimization (second-order methods' linear systems, `04.07`; certain regularized least-squares
problems), this difference between $O(n^3)$ and "a modest number of cheap matrix-vector
products" is often what separates feasible from infeasible.

---

In [ ]:
# Setup
import numpy as np
from scipy.sparse.linalg import cg as scipy_cg
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Tracking CG's residual norm across iterations for systems with increasing condition number
shows convergence slowing down smoothly with $\kappa$ — never breaking the hard $n$-iteration
guarantee, but taking noticeably longer to get close for worse-conditioned systems.

In [ ]:
def conjugate_gradient_track(A, b, tol=1e-10, max_iter=None):
    n = len(b)
    max_iter = max_iter or n
    x = np.zeros(n)
    r = b - A @ x
    p = r.copy()
    rs_old = r @ r
    residual_norms = [np.sqrt(rs_old)]
    for _ in range(max_iter):
        Ap = A @ p
        alpha = rs_old / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = r @ r
        residual_norms.append(np.sqrt(rs_new))
        if np.sqrt(rs_new) < tol:
            break
        p = r + (rs_new / rs_old) * p
        rs_old = rs_new
    return x, residual_norms

rng = np.random.default_rng(0)
n = 50
fig, ax = plt.subplots(figsize=(8, 5.5))
for kappa_target, color in zip([2, 10, 100, 1000], ["#55A868", "#4C72B0", "#DD8452", "#C44E52"]):
    eigvals = np.linspace(1, kappa_target, n)
    Q, _ = np.linalg.qr(rng.normal(size=(n, n)))
    A_cond = Q @ np.diag(eigvals) @ Q.T   # same construction as 04.02/04.03's conditioning demos
    b_cond = rng.normal(size=n)
    _, residuals = conjugate_gradient_track(A_cond, b_cond, tol=1e-10, max_iter=n)
    ax.semilogy(residuals, color=color, lw=2, label=f"kappa={kappa_target}")

ax.set_xlabel("CG iteration"); ax.set_ylabel("residual norm ||Ax - b|| (log scale)")
ax.set_title(f"CG convergence speed vs. condition number\n(all bounded by n={n} iterations, per the theoretical guarantee)")
ax.legend()
plt.show()

print("Every curve reaches machine precision well before n=50 iterations -- but higher kappa")
print("visibly takes longer per decade of residual reduction, matching the ~sqrt(kappa) rate.")

## 🐍 Implementation from Scratch

We implement Conjugate Gradient from scratch, confirm it matches a direct solve and
`scipy.sparse.linalg.cg`, verify the at-most-$n$-iterations guarantee, and measure how
condition number affects convergence speed.

In [ ]:
def conjugate_gradient(A, b, tol=1e-10, max_iter=None):
    n = len(b)
    max_iter = max_iter or n   # the theoretical guarantee: never need more than n iterations
    x = np.zeros(n)
    r = b - A @ x
    p = r.copy()
    rs_old = r @ r
    for i in range(max_iter):
        Ap = A @ p
        alpha = rs_old / (p @ Ap)        # exact step size along the current A-conjugate direction
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = r @ r
        if np.sqrt(rs_new) < tol:
            return x, i + 1
        p = r + (rs_new / rs_old) * p    # next direction, A-conjugate to all previous ones
        rs_old = rs_new
    return x, max_iter

rng = np.random.default_rng(0)
n = 20
M = rng.normal(size=(n, n))
A = M @ M.T + n * np.eye(n)  # M @ M.T is always PSD; adding n*I guarantees strict positive-definiteness
b = rng.normal(size=n)

x_cg, n_iters = conjugate_gradient(A, b)
x_direct = np.linalg.solve(A, b)
print(f"CG converged in {n_iters} iterations (n={n}; guarantee: at most {n})")
print(f"CG matches direct solve (np.linalg.solve): {np.allclose(x_cg, x_direct, atol=1e-6)}")

x_scipy, _ = scipy_cg(A, b, rtol=1e-10)
print(f"CG matches scipy.sparse.linalg.cg:         {np.allclose(x_cg, x_scipy, atol=1e-6)}")

# --- Condition number vs. convergence speed ---
print(f"\n{'kappa':>8} {'iters to shrink residual 1e-4x':>32}")
for kappa_target in [2, 10, 100, 1000]:
    eigvals = np.linspace(1, kappa_target, n)
    Q, _ = np.linalg.qr(rng.normal(size=(n, n)))
    A_k = Q @ np.diag(eigvals) @ Q.T
    b_k = rng.normal(size=n)
    x0 = np.zeros(n)
    r0_norm = np.linalg.norm(b_k - A_k @ x0)
    x_k = x0.copy()
    r = b_k - A_k @ x_k
    p = r.copy()
    rs_old = r @ r
    for i in range(n):
        Ap = A_k @ p
        alpha = rs_old / (p @ Ap)
        x_k = x_k + alpha * p
        r = r - alpha * Ap
        rs_new = r @ r
        if np.sqrt(rs_new) < r0_norm * 1e-4:
            print(f"{kappa_target:8d} {i+1:32d}")
            break
        p = r + (rs_new / rs_old) * p
        rs_old = rs_new

## 🤖 Why This Matters for AI

Whenever a large-scale ML system needs to solve a linear system exactly rather than
approximately — natural gradient methods forming $F^{-1}\nabla L$ (recall `04.07`'s K-FAC
discussion), certain Gaussian process computations, or large sparse least-squares problems —
conjugate gradient (or a close relative) is typically what's actually running underneath,
because forming and inverting the full matrix directly is too expensive at scale.
**Preconditioning** is the practical lever used to make CG converge fast enough to be useful:
below, we build the simplest possible preconditioner (a **Jacobi preconditioner**, just the
matrix's own diagonal) for a poorly-scaled system and measure directly how many fewer
iterations CG needs with it.

In [ ]:
# A poorly-SCALED SPD system: diagonal entries span a huge range, which is exactly the kind
# of ill-conditioning a cheap diagonal (Jacobi) preconditioner is designed to fix.
n_precond = 100
D = np.diag(rng.uniform(0.01, 100, size=n_precond))
M_rand = rng.normal(scale=0.1, size=(n_precond, n_precond))
A_precond = D + M_rand @ M_rand.T
b_precond = rng.normal(size=n_precond)

count_none = {"n": 0}
x_none, info_none = scipy_cg(A_precond, b_precond, rtol=1e-8,
                              callback=lambda xk: count_none.__setitem__("n", count_none["n"] + 1))

jacobi_preconditioner = np.diag(1.0 / np.diag(A_precond))  # M^-1 = diag(1/A_ii): cheap to build AND apply
count_jacobi = {"n": 0}
x_jacobi, info_jacobi = scipy_cg(A_precond, b_precond, rtol=1e-8, M=jacobi_preconditioner,
                                   callback=lambda xk: count_jacobi.__setitem__("n", count_jacobi["n"] + 1))

print(f"System size: n={n_precond}")
print(f"No preconditioner:     {count_none['n']} iterations to converge")
print(f"Jacobi preconditioner: {count_jacobi['n']} iterations to converge")
print(f"Solutions agree: {np.allclose(x_none, x_jacobi, atol=1e-4)}")

cond_before = np.linalg.cond(A_precond)
cond_after = np.linalg.cond(jacobi_preconditioner @ A_precond)
print(f"\ncondition number BEFORE preconditioning: {cond_before:.1f}")
print(f"condition number AFTER preconditioning:  {cond_after:.1f}")
print(f"\nA preconditioner this cheap (just the matrix's own diagonal) bought a")
print(f"{count_none['n']/count_jacobi['n']:.1f}x reduction in iterations -- exactly by shrinking kappa,")
print(f"which the Theory section's sqrt(kappa) convergence-rate scaling predicts should help this much.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $A=\begin{pmatrix}4&1\\1&3\end{pmatrix}$, $\mathbf{b}=(1,2)$, run `conjugate_gradient`
   by hand starting from $\mathbf{x}_0=\mathbf{0}$: compute $\mathbf{r}_0, \mathbf{p}_0$,
   $\alpha_0$, $\mathbf{x}_1$, then $\mathbf{x}_2$. Confirm $\mathbf{x}_2$ matches
   `np.linalg.solve(A, b)` exactly — the guaranteed outcome for a $2\times 2$ SPD system, which
   must converge in at most 2 iterations.

<details>
<summary>💡 Solution</summary>

$\mathbf{r}_0 = \mathbf{b} = (1,2)$, $\mathbf{p}_0=(1,2)$, $\alpha_0 = \frac{\mathbf{r}_0^\top\mathbf{r}_0}{\mathbf{p}_0^\top A\mathbf{p}_0} = \frac{5}{20}=0.25$,
giving $\mathbf{x}_1=(0.25, 0.5)$ and $\mathbf{r}_1=(-0.5,0.25)$. A second step with
$\alpha_1\approx0.3636$ gives $\mathbf{x}_2 \approx (0.0909, 0.6364)$ with residual norm
exactly $0$ (to machine precision) — matching `np.linalg.solve(A,b)` exactly, confirming the
2-iteration guarantee for this $2\times 2$ system.

</details>

### 💭 UNDERSTAND
2. `conjugate_gradient`'s search directions $\mathbf{p}_i$ are constructed to be
   **A-conjugate**: $\mathbf{p}_i^\top A \mathbf{p}_j = 0$ for $i\ne j$. Someone claims this is
   just a restatement of the directions being *orthogonal* ($\mathbf{p}_i^\top \mathbf{p}_j=0$).
   Are these the same condition? If not, construct (or reason through) a case where they differ.

<details>
<summary>💡 Solution</summary>

They are not the same condition — A-conjugacy is orthogonality *with respect to the inner
product $\langle u,v\rangle_A = u^\top A v$*, not the standard inner product, and the two only
coincide when $A$ is a multiple of the identity. For a general SPD $A$ with distinct eigenvalues,
two vectors can be perfectly A-conjugate while having a nonzero standard dot product (and vice
versa) — running CG's own tracked directions on a random SPD matrix and checking both
$\mathbf{p}_i^\top A\mathbf{p}_j$ (essentially zero) and $\mathbf{p}_i^\top \mathbf{p}_j$
(generally nonzero) for $i\ne j$ confirms this directly. This distinction is the entire point
of CG's construction: it deliberately chooses the inner product in which $A$ itself defines
"perpendicular," which is what makes each step permanently eliminate one component of the error
with respect to $A$, not with respect to the ordinary geometry of $\mathbb{R}^n$.

</details>

### ✏️ DERIVE
3. `conjugate_gradient`'s stated guarantee is convergence in at most $n$ iterations for an
   $n\times n$ SPD system. Sketch why: show that the set of search directions
   $\{\mathbf{p}_0,\ldots,\mathbf{p}_{n-1}\}$, being mutually A-conjugate and nonzero, must be
   linearly independent, and explain why $n$ linearly independent directions in $\mathbb{R}^n$
   are enough to reach any point exactly (including the true solution) via some combination of
   steps along them.

<details>
<summary>💡 Solution outline</summary>

Suppose $\sum_i c_i \mathbf{p}_i = 0$ for scalars $c_i$. Take the $A$-inner-product with any
$\mathbf{p}_j$: $\sum_i c_i (\mathbf{p}_i^\top A \mathbf{p}_j) = 0$. By A-conjugacy, every term
with $i\ne j$ vanishes, leaving $c_j(\mathbf{p}_j^\top A\mathbf{p}_j) = 0$. Since $A$ is
positive-definite and $\mathbf{p}_j\ne 0$, $\mathbf{p}_j^\top A\mathbf{p}_j > 0$, forcing
$c_j=0$ for every $j$ — so the directions are linearly independent. $n$ linearly independent
vectors in $\mathbb{R}^n$ form a basis, so the true error $\mathbf{x}^* - \mathbf{x}_0$ can be
written exactly as $\sum_i \alpha_i \mathbf{p}_i$ for some scalars; CG's step-size choice at
each iteration is exactly the $\alpha_i$ needed to eliminate that direction's component of the
error, so after all $n$ steps the full error is cancelled and $\mathbf{x}_n = \mathbf{x}^*$
exactly (in exact arithmetic).

</details>

### 🐍 IMPLEMENT
4. Modify the condition-number sweep to also record, for each $\kappa$ tested, the ratio
   (iterations to shrink the residual $10^{-4}\times$) $/ \sqrt{\kappa}$. Run it at $n=20$ (as
   in the notebook) and separately at a larger $n=200$, and compare.

<details>
<summary>✅ How to know you're right</summary>

At $n=20$, the ratio is *not* constant across $\kappa\in\{2,10,100,1000\}$ — it actually
*decreases* at large $\kappa$ (drops to about $0.63$ at $\kappa=1000$), because the hard
$n$-iteration cap is binding: the system can't take more than 20 steps regardless of how slow
$\sqrt{\kappa}$-scaling would want it to converge, so the observed iteration count is
artificially suppressed below what the asymptotic rate predicts. At $n=200$ (large enough that
the cap doesn't bind for these $\kappa$ values), the ratio is much more stable across
$\kappa$ (roughly $4$–$4.4$ for $\kappa\in\{2,10,100\}$, only dropping off at $\kappa=1000$
where 65 iterations is still comfortably under the 200-iteration cap) — matching the
$\sqrt{\kappa}$ scaling law much better once the hard cap isn't interfering. The exercise's
real lesson: the exact $n$-step guarantee and the $\sqrt{\kappa}$ convergence-rate heuristic are
two different regimes, and small $n$ can hide the second one.

</details>

### 🤖 APPLY
5. A Gaussian process regression model needs to solve $(K + \sigma^2 I)\boldsymbol{\alpha} =
   \mathbf{y}$ for the kernel (Gram) matrix $K$ (recall `08.01`'s Gram matrices) on $n=5000$
   training points — too large to invert directly at $O(n^3)$ cost. Explain why CG is a natural
   fit here specifically (what property of $K+\sigma^2 I$ makes it usable), and why adding
   $\sigma^2 I$ (the observation-noise term already present in the GP model, not an artificial
   addition) is doubly helpful for CG's convergence speed beyond its statistical role.

<details>
<summary>✅ What you should observe</summary>

$K$ is a Gram matrix, hence symmetric PSD (recall `08.01`); adding $\sigma^2 I$ for $\sigma^2>0$
makes it strictly positive-definite, exactly CG's required SPD condition — so CG applies
directly, needing only matrix-vector products with $K+\sigma^2 I$ (never forming or inverting
the full matrix). The $\sigma^2 I$ term is doubly useful: statistically it's the observation
noise variance, but numerically it also *shifts every eigenvalue of $K$ up by $\sigma^2$*,
which raises $\lambda_{min}$ and therefore shrinks the condition number
$\kappa=(\lambda_{max}+\sigma^2)/(\lambda_{min}+\sigma^2)$ relative to $K$ alone — the same
mechanism as `04.03`'s L2-regularization-improves-conditioning argument, now appearing as a
byproduct of a modeling choice rather than an explicit optimization regularizer.

</details>

### 🚀 CHALLENGE
6. Construct a *worse*-conditioned version of the Jacobi-preconditioning system (widen the
   `rng.uniform` diagonal-scale range from `(0.01, 100)` to `(0.0001, 10000)`) and compare the
   Jacobi preconditioner's relative speedup (iterations without / iterations with) between the
   original and worse-conditioned systems. Does the speedup grow, shrink, or stay about the
   same? Connect your answer to why more sophisticated preconditioners (incomplete Cholesky,
   multigrid) exist beyond the simple diagonal one used here.

<details>
<summary>🔍 What a strong answer covers</summary>

Numerically: the original system has $\kappa\approx74.8$ before preconditioning and
$\kappa\approx2.1$ after (Jacobi speedup $\approx6.9\times$, from 55 to 8 iterations); the
worse-conditioned system starts at $\kappa\approx156.7$ but the Jacobi preconditioner brings it
all the way down to $\kappa\approx1.0$ (nearly the identity), giving a *larger* speedup
($\approx19\times$, from 57 down to just 3 iterations). The speedup grows because this
particular system's ill-conditioning comes almost entirely from the diagonal $D$'s huge dynamic
range — precisely the structure a diagonal (Jacobi) preconditioner is built to cancel, so the
worse that specific structure gets, the more relative good a diagonal fix does. A strong answer
notes this is a best case for Jacobi *by construction* and doesn't generalize: if ill-conditioning
instead comes from near-linear-dependence between off-diagonal directions (which a diagonal
preconditioner cannot see or fix), Jacobi would show little or no improvement — which is exactly
why incomplete Cholesky (captures some off-diagonal structure cheaply) and multigrid (captures
smooth error components across multiple scales) exist as preconditioners: different
ill-conditioning structures need preconditioners that target that specific structure, and no
single cheap preconditioner is uniformly good.

</details>

---

## 📚 Further Reading
- Shewchuk, ["An Introduction to the Conjugate Gradient Method Without the Agonizing Pain"](https://www.cs.cmu.edu/~quake-papers/painless-conjugate-gradient.pdf)
- Nocedal & Wright, *Numerical Optimization*, Ch. 5 (Conjugate Gradient Methods)

---

*Next notebook: [Interpolation and Approximation](05_interpolation_and_approximation.ipynb)*